# Project 10 — Compact, Reproducible Modeling Notebook

**Research question:** multiclass network-intrusion classification under severe class imbalance.

This notebook contains only the final data preparation, leakage-aware train/test split, selected model, held-out test evaluation, and export steps. Exploratory benchmarks, SMOTE/weighted-model experiments, repeated plots, cross-validation loops, and hyperparameter searches have been removed from the routine run. Their selected outcome is documented below; the expensive experiments can remain in a separate research/experiments notebook.

**Selected model:** class-weighted Random Forest with the hyperparameters selected during the completed development search (`n_estimators=200`, `max_depth=40`, `min_samples_split=10`, `min_samples_leaf=2`, `max_features='sqrt'`). The previous grouped randomized search reported best CV Macro F1 ≈ **0.5502**. This compact notebook uses those fixed settings and does **not** repeat the search.

**Important:** this notebook expects the same `merged_dataset.csv` version used in the completed experiment. Dataset provenance and earlier EDA row-count discrepancies must be reconciled before publication. Do not silently substitute another dataset version.


## 1. Environment and configuration
Run the install cell only if the runtime is missing a dependency. Google Colab usually already includes the scientific Python stack.

In [ ]:
# If needed, install only the dependency used by this final workflow.
%pip -q install imbalanced-learn

import os
import time
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import imblearn

from google.colab import drive
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)
import joblib

SEED = 42
LABEL_COL = "Label"
DATA_PATH = "/content/drive/MyDrive/merged_dataset.csv"

CLASS_NAMES = [
    "Benign", "Analysis", "Backdoor", "DoS", "Exploits",
    "Fuzzers", "Generic", "Reconnaissance", "Shellcode", "Worms"
]
print("Python packages:", {"pandas": pd.__version__, "numpy": np.__version__,
                            "scikit-learn": sklearn.__version__, "imbalanced-learn": imblearn.__version__})


## 2. Load and minimally validate the dataset
The dataset is loaded once. Exact duplicate rows are removed using the same operation as the completed experiment. We retain identical feature vectors with conflicting labels rather than arbitrarily relabeling them; grouping by features keeps such vectors from crossing the train/test boundary.

In [ ]:
drive.mount("/content/drive", force_remount=False)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Upload merged_dataset.csv to Google Drive "
        "and update DATA_PATH if it is stored elsewhere."
    )

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
if LABEL_COL not in df.columns:
    raise ValueError(f"Required target column {LABEL_COL!r} was not found.")

rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows before exact deduplication: {rows_before:,}")
print(f"Rows after exact deduplication:  {len(df):,}")
print(f"Exact duplicate rows removed:   {rows_before - len(df):,}")

X = df.drop(columns=[LABEL_COL])
y = df[LABEL_COL].astype(int)
if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in X.dtypes):
    raise TypeError("This final model was developed for numerical predictors only; inspect non-numeric columns.")
if not set(y.unique()).issubset(set(range(10))):
    raise ValueError("Unexpected target labels. Expected integer labels 0–9.")

print("Feature matrix:", X.shape)
print("Target counts:
", y.value_counts().sort_index())
print("Missing feature values:", int(X.isna().sum().sum()))
print("Infinite numeric values:", int(np.isinf(X.to_numpy(dtype=float)).sum()))


## 3. Leakage-aware train/test split
A hash of each feature vector is used as its group ID. The first split from a shuffled, seeded `StratifiedGroupKFold` is retained to match the original workflow. The test partition is held out until final evaluation.

In [ ]:
groups = pd.util.hash_pandas_object(X, index=False)

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

overlap = set(groups_train).intersection(set(groups_test))
if overlap:
    raise RuntimeError(f"Leakage check failed: {len(overlap)} feature groups cross the split.")

print("Train shape:", X_train.shape)
print("Test shape: ", X_test.shape)
print("Feature-group overlap:", len(overlap))
print("Test class distribution:
", y_test.value_counts(normalize=True).sort_index().round(4))


## 4. Build and fit the selected final model
The preprocessing step is kept inside the pipeline so any imputation is learned from training data only. Since the predictors are numerical and the selected estimator is tree-based, no scaling or categorical encoding is required.

In [ ]:
numeric_features = X_train.columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[("numeric", SimpleImputer(strategy="median"), numeric_features)],
    remainder="drop"
)

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=40,
        min_samples_split=10,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ))
])

start = time.perf_counter()
final_model.fit(X_train, y_train)
training_time = time.perf_counter() - start
print(f"Final model fit completed in {training_time:.2f} seconds")


## 5. One-time final test evaluation
These metrics are computed once on the held-out test set. Do not use this test set to tune parameters or select another model.

In [ ]:
start = time.perf_counter()
y_test_pred = final_model.predict(X_test)
prediction_time = time.perf_counter() - start

final_metrics = {
    "Model": "Tuned Balanced Random Forest",
    "Accuracy": float(accuracy_score(y_test, y_test_pred)),
    "Macro Precision": float(precision_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "Macro Recall": float(recall_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "Macro F1": float(f1_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "Weighted F1": float(f1_score(y_test, y_test_pred, average="weighted", zero_division=0)),
    "Training Time (s)": float(training_time),
    "Prediction Time (s)": float(prediction_time),
    "Test Samples": int(len(y_test))
}

display(pd.Series(final_metrics, name="Final test result").to_frame().round(4))
print(classification_report(
    y_test, y_test_pred, labels=list(range(10)), target_names=CLASS_NAMES,
    zero_division=0, digits=4
))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred,
    labels=list(range(10)),
    display_labels=CLASS_NAMES,
    normalize="true",
    xticks_rotation=45,
    values_format=".2f",
    cmap="Blues"
)
plt.title("Normalized Confusion Matrix — Final Test Set")
plt.tight_layout()
plt.show()


## 6. Save model and final results
The saved model includes preprocessing. These files let you reuse the trained model and inspect the final predictions without retraining it. Keep the dataset private or comply with its license when sharing outputs.

In [ ]:
joblib.dump(final_model, OUTPUT_DIR / "tuned_balanced_random_forest.joblib")

pd.DataFrame({"y_true": y_test.to_numpy(), "y_pred": y_test_pred}).to_csv(
    OUTPUT_DIR / "final_test_predictions.csv", index=False
)
with open(OUTPUT_DIR / "final_test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, indent=4)

run_metadata = {
    "seed": SEED,
    "dataset_path": DATA_PATH,
    "dataset_rows_after_deduplication": int(len(df)),
    "feature_count": int(X.shape[1]),
    "splitter": "StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42), first split",
    "grouping": "pandas hash of feature vectors; zero train/test group overlap verified",
    "model": "RandomForestClassifier",
    "model_parameters": final_model.named_steps["model"].get_params(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
    "imbalanced_learn_version": imblearn.__version__
}
with open(OUTPUT_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=4, default=str)

print(f"Saved final model and evaluation files to: {OUTPUT_DIR}")
print("Files:", sorted(p.name for p in OUTPUT_DIR.iterdir()))


## Reproducibility notes and limitations

- This notebook intentionally skips the completed baseline, class-weighting/SMOTE comparisons, advanced-model experiments, grouped CV leaderboard, randomized searches, development error analysis, and duplicate validation plots. Keep those in an archive/experiments notebook if needed for auditability.
- The fixed parameters above are taken from the completed Balanced Random Forest search. If you change them or change the data/split, this is a new experiment and the final metrics may change.
- A fresh run refits the selected model and reevaluates the test set. To inspect/use an already saved model without retraining, load `outputs/tuned_balanced_random_forest.joblib` in a runtime where that file is available.
- The earlier EDA notes and this modeling notebook have documented different intermediate row counts. Verify the exact merged dataset, source-file alignment, and duplicate-removal history before publishing a paper.
- Results are specific to this dataset and split; they do not alone establish generalization to other networks or future traffic.
